In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image



In [ ]:
import torchvision
import torchvision.transforms as transforms

# Download and load training and test sets
trainset = torchvision.datasets.CIFAR10(root='Datasets/cifar10', train=True, download=True)
testset = torchvision.datasets.CIFAR10(root='Datasets/cifar10', train=False, download=True)


46.0%

In [ ]:
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def get_device():
    """Check if GPU is available and return the appropriate device for MPS, CUDA, or CPU."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif torch.backends.mps.is_available() and torch.backends.mps.is_built():
        return torch.device("mps")
    else:
        return torch.device("cpu")

def 

In [ ]:


# Set to your image path, e.g. "/path/to/photo.jpg"
IMAGE_PATH = None
IMG_SIZE = 224  # resize target (must be divisible by PATCH_SIZE)

if IMAGE_PATH and Path(IMAGE_PATH).exists():
    img_pil = Image.open(IMAGE_PATH).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
else:
    # synthetic test image: smooth colour gradients so patches are visually distinct
    x = np.linspace(0, 255, IMG_SIZE, dtype=np.uint8)
    r = np.tile(x, (IMG_SIZE, 1))
    g = np.tile(x[:, None], (1, IMG_SIZE))
    b = np.full((IMG_SIZE, IMG_SIZE), 128, dtype=np.uint8)
    img_pil = Image.fromarray(np.stack([r, g, b], axis=-1))

img = np.array(img_pil)

plt.figure(figsize=(4, 4))
plt.imshow(img)
plt.title("Original image")
plt.axis("off")
plt.tight_layout()
plt.show()
print(f"Shape: {img.shape}")

In [ ]:
PATCH_SIZE = 16

H, W, C = img.shape
assert H % PATCH_SIZE == 0 and W % PATCH_SIZE == 0

n_rows = H // PATCH_SIZE
n_cols = W // PATCH_SIZE
gap = 2  # white border between patches (pixels)

grid_h = n_rows * PATCH_SIZE + (n_rows + 1) * gap
grid_w = n_cols * PATCH_SIZE + (n_cols + 1) * gap
grid = np.full((grid_h, grid_w, 3), 255, dtype=np.uint8)

for i in range(n_rows):
    for j in range(n_cols):
        y = gap + i * (PATCH_SIZE + gap)
        x = gap + j * (PATCH_SIZE + gap)
        grid[y : y + PATCH_SIZE, x : x + PATCH_SIZE] = img[
            i * PATCH_SIZE : (i + 1) * PATCH_SIZE,
            j * PATCH_SIZE : (j + 1) * PATCH_SIZE,
        ]

plt.figure(figsize=(8, 8))
plt.imshow(grid)
plt.title(f"{n_rows}x{n_cols} patches of {PATCH_SIZE}x{PATCH_SIZE}  ({n_rows * n_cols} total)")
plt.axis("off")
plt.tight_layout()
plt.show()